# NSYS 2026: PINN HPO Metaheuristic Comparison
## GA vs PSO vs ACO vs Fuzzy-GA vs Fuzzy-PSO vs Fuzzy-ACO

Compares six metaheuristic optimizers for Physics-Informed Neural Network (PINN)
hyperparameter tuning across four PDE benchmarks (ODE, Heat, Burgers, Wave) with
real PyTorch training - not placeholder/synthetic results.

**What this notebook does:**
- Runs all 6 algorithms x 4 benchmarks x N seeds (parallelized across cores)
- Tracks real per-generation population/swarm/archive diversity for every algorithm
  (not just the fuzzy-adaptive ones)
- Generates convergence + diversity plots, a markdown report, and an ACM sigconf
  (two-column) LaTeX paper skeleton
- Packages everything into a downloadable zip

Runtime: roughly 2-4 hours for the full run (2-3 seeds), or ~20-30 min in quick mode.


## Step 1: Install dependencies

In [ ]:
!pip install -q torch numpy scipy matplotlib tqdm
print("Dependencies installed.")


## Step 2: Clone the repository

In [ ]:
import os

if not os.path.isdir('OptimizationOverviewPINN'):
    !git clone -q https://github.com/Rahuldrabit/OptimizationOverviewPINN.git
os.chdir('OptimizationOverviewPINN')

print("Repository ready at:", os.getcwd())
!ls scripts | grep nsys


## Step 3: Configure the run

- `quick_mode = True` runs a fast smoke test (small population/generations, ~20-30 min) -
  useful to confirm everything works before committing to the full run.
- `quick_mode = False` runs the real manuscript-scale benchmark (1200 training steps
  per candidate, ~2-4 hours depending on `n_seeds` and Colab's CPU allocation).


In [ ]:
import numpy as np  # imported once here, available in every later cell

# ==== CONFIGURATION ====
quick_mode = False        # True = fast smoke test, False = full manuscript run
n_seeds = 2                # 2-3 recommended for the full run
n_steps = 100 if quick_mode else 1200
max_workers = 4            # Colab typically gives 2-4 vCPUs
output_dir = "outputs/nsys2026_colab"

benchmarks = ["ode", "heat", "burgers", "wave"]
algorithms = ["GA", "PSO", "ACO", "Fuzzy-GA", "Fuzzy-PSO", "Fuzzy-ACO"]
total_jobs = len(benchmarks) * len(algorithms) * n_seeds

print("Configuration:")
print(f"  Mode          : {'QUICK SMOKE TEST' if quick_mode else 'FULL MANUSCRIPT RUN'}")
print(f"  Seeds         : {n_seeds}")
print(f"  Steps/eval    : {n_steps}")
print(f"  Workers       : {max_workers}")
print(f"  Benchmarks    : {benchmarks}")
print(f"  Algorithms    : {algorithms}")
print(f"  Total jobs    : {total_jobs}")
print(f"  Output dir    : {output_dir}")


## Step 4: Run the benchmark

In [ ]:
import sys
sys.path.insert(0, "src")

from hpo.comparison import ExperimentConfig, run_experiment_grid
from hpo.report_generator import generate_all_plots, generate_markdown_report
from utils import ensure_dir

config = ExperimentConfig(
    benchmarks=benchmarks,
    algorithms=algorithms,
    seeds=list(range(n_seeds)),
    n_steps=n_steps,
    output_dir=output_dir,
)

print("=" * 70)
print(f"LAUNCHING BENCHMARK ({total_jobs} total jobs, {max_workers} parallel workers)")
print("=" * 70)

results = run_experiment_grid(config, quick=quick_mode, verbose=True, max_workers=max_workers)

print("\nAll jobs complete.")


## Step 5: Generate plots and report

In [ ]:
plots_dir = os.path.join(config.output_dir, "plots")
ensure_dir(plots_dir)

print("[+] Generating plots...")
plot_files = generate_all_plots(results, plots_dir)
for name, path in plot_files.items():
    print(f"    - {name}: {path}")

report_file = os.path.join(config.output_dir, "MANUSCRIPT_REPORT.md")
print(f"\n[+] Writing report to '{report_file}'...")
generate_markdown_report(results, plot_files, report_file)

print("\n" + "=" * 70)
print("RANKINGS")
print("=" * 70)
for rank, (alg, data) in enumerate(results["overall_rankings"].items(), start=1):
    div = data.get("overall_mean_diversity", float("nan"))
    div_str = f"{div:.3f}" if not np.isnan(div) else "n/a"
    print(f"  #{rank}: {alg:12s} | Avg Rank: {data['average_rank']:.2f} | "
          f"Mean L2: {data['overall_mean_rel_l2']:.6f} | Mean Diversity: {div_str}")
print("=" * 70)


## Step 6: Preview the plots inline

In [ ]:
from IPython.display import Image, display

for name in ["convergence", "diversity", "performance", "radar", "heatmap"]:
    if name in plot_files:
        print(f"\n--- {name.upper()} ---")
        display(Image(filename=plot_files[name]))


## Step 7: Generate the ACM sigconf (two-column) paper

Every number in the generated `.tex` file is pulled directly from the results computed
above - nothing here is a hardcoded or assumed constant.

In [ ]:
import json as _json

def compute_algo_stats(results_data, algo_list):
    """Aggregate mean/std error, runtime, and mean diversity per algorithm across
    all benchmarks/seeds. All numbers come directly from raw_runs."""
    raw_runs = results_data.get("raw_runs", {})
    bmarks = results_data.get("metadata", {}).get("benchmarks", [])
    stats = {}
    for algo in algo_list:
        errs, times, diversities = [], [], []
        for bmark in bmarks:
            for r in raw_runs.get(bmark, {}).get(algo, []):
                errs.append(r["val_rel_l2"])
                times.append(r["runtime_sec"])
                dh = r.get("diversity_history", [])
                if dh:
                    diversities.extend(step["diversity"] for step in dh)
        if errs:
            stats[algo] = {
                "mean_l2": float(np.mean(errs)),
                "std_l2": float(np.std(errs)),
                "min_l2": float(np.min(errs)),
                "max_l2": float(np.max(errs)),
                "mean_time": float(np.mean(times)),
                "mean_diversity": float(np.mean(diversities)) if diversities else float("nan"),
                "n_runs": len(errs),
            }
    return stats


def build_latex(results_data, stats, figures_rel="../plots"):
    bmarks = results_data.get("metadata", {}).get("benchmarks", [])
    seeds = results_data.get("metadata", {}).get("seeds", [])
    sorted_algos = sorted(stats.items(), key=lambda kv: kv[1]["mean_l2"])
    best_alg, best_stats = sorted_algos[0]
    worst_alg, worst_stats = sorted_algos[-1]

    sig_note = (
        f"With only {len(seeds)} random seed(s) per (algorithm, benchmark) cell, this "
        r"study is \emph{not} statistically powered for a formal significance test; "
        "rank differences should be read as descriptive, not as evidence of a "
        r"significant effect. A follow-up with $\geq$10 seeds is needed before any "
        "claim of statistical significance."
    )

    lines = []
    lines.append(r"\documentclass[sigconf]{acmart}")
    lines.append(r"\usepackage{amsmath,amssymb,amsfonts}")
    lines.append(r"\usepackage{graphicx}")
    lines.append(r"\usepackage{booktabs}")
    lines.append(r"\settopmatter{printacmref=false}")
    lines.append(r"\renewcommand\footnotetextcopyrightpermission[1]{}")
    lines.append(r"\pagestyle{plain}")
    lines.append("")
    lines.append(r"\title{An Empirical Comparison of Genetic, Particle Swarm, Ant Colony, "
                 r"and Fuzzy-Adaptive Metaheuristics for Physics-Informed Neural Network "
                 r"Hyperparameter Optimization}")
    lines.append("")
    lines.append(r"\author{Rahul Drabit Chowdhury}")
    lines.append(r"\affiliation{\institution{Independent Research}\country{}}")
    lines.append("")
    lines.append(r"\begin{document}")
    lines.append(r"\begin{abstract}")
    lines.append(
        "We empirically compare six metaheuristic hyperparameter optimizers -- Genetic "
        "Algorithm (GA), Particle Swarm Optimization (PSO), Ant Colony Optimization "
        "(ACO/ACOR), and their Mamdani fuzzy-adaptive variants (Fuzzy-GA, Fuzzy-PSO, "
        f"Fuzzy-ACO) -- for tuning Physics-Informed Neural Networks (PINNs) across "
        f"{len(bmarks)} PDE benchmarks ({', '.join(b.upper() for b in bmarks)}) with "
        f"{len(seeds)} random seed(s) per configuration. We log real per-generation "
        "population/swarm/archive diversity for every algorithm (not only the fuzzy "
        "variants) to characterize exploration-exploitation behavior directly, rather "
        f"than assuming it from algorithm category. The best-performing method, "
        f"{best_alg}, reaches a mean relative $L_2$ error of {best_stats['mean_l2']:.6f} "
        f"(min {best_stats['min_l2']:.6f}) versus {worst_stats['mean_l2']:.6f} for the "
        f"weakest, {worst_alg}. {sig_note}"
    )
    lines.append(r"\end{abstract}")
    lines.append("")
    lines.append(r"\maketitle")
    lines.append("")
    lines.append(r"\section{Introduction}")
    lines.append(
        "Physics-Informed Neural Networks (PINNs) embed PDE residuals directly into the "
        "training loss, but their accuracy is highly sensitive to hyperparameters "
        "(architecture, activation, optimizer, learning rate, and physics/initial-condition "
        "loss weights). This paper reports a controlled, same-codebase comparison of six "
        "classical and fuzzy-adaptive metaheuristics under an identical search space and "
        "training budget, with particular attention to two properties that prior "
        r"comparisons typically assume rather than measure: population/swarm/archive "
        r"\emph{diversity} across generations, and how that diversity trades off against "
        "convergence speed."
    )
    lines.append("")
    lines.append(r"\section{Methodology}")
    lines.append(r"\subsection{Algorithms}")
    lines.append(r"\begin{itemize}")
    lines.append(r"    \item \textbf{GA}: tournament selection, single-point crossover, elitist replacement.")
    lines.append(r"    \item \textbf{PSO}: inertia-weighted velocity update with cognitive/social terms.")
    lines.append(r"    \item \textbf{ACO / ACOR}: continuous ant colony optimization with a Gaussian-kernel-weighted solution archive.")
    lines.append(r"    \item \textbf{Fuzzy-GA / Fuzzy-PSO / Fuzzy-ACO}: each classical algorithm augmented with a Mamdani fuzzy inference controller that adapts mutation rate, inertia, or archive spread from measured population diversity and fitness improvement each generation.")
    lines.append(r"\end{itemize}")
    lines.append("")
    lines.append(r"\subsection{Diversity / Exploration Measurement}")
    lines.append(
        "For every algorithm (not only the fuzzy variants), we compute a normalized "
        "diversity score each generation/iteration as the mean Euclidean distance of the "
        "population/swarm/archive to its own centroid in the normalized search space, "
        "divided by the theoretical maximum spread of a unit hypercube of the same "
        "dimensionality - a directly comparable, measured exploration signal across all "
        "six algorithms."
    )
    lines.append("")
    lines.append(r"\subsection{Search Space and Benchmarks}")
    lines.append(
        "All algorithms optimize the same 8-dimensional space: hidden layers, hidden "
        "width, activation (tanh / sine / swish), optimizer (Adam / AdamW / L-BFGS), "
        "learning rate (log-scale), physics loss weight, initial-condition loss weight, "
        f"and number of collocation points, evaluated on {len(bmarks)} PDE benchmarks with "
        f"real PyTorch training: {', '.join(b.upper() for b in bmarks)}."
    )
    lines.append("")
    lines.append(r"\section{Results}")
    lines.append(r"\subsection{Overall Ranking}")
    lines.append(r"\begin{table}[h]")
    lines.append(r"\centering")
    lines.append(r"\small")
    lines.append(
        r"\caption{Aggregate performance across " + str(len(bmarks)) + r" benchmarks $\times$ "
        + str(len(seeds)) + r" seed(s). Diversity is the measured mean normalized "
        r"population/swarm/archive spread across all recorded generations.}"
    )
    lines.append(r"\begin{tabular}{lccc}")
    lines.append(r"\toprule")
    lines.append(r"\textbf{Algorithm} & \textbf{Mean $L_2$} & \textbf{Std $L_2$} & \textbf{Mean Diversity} \\")
    lines.append(r"\midrule")
    for algo, s in sorted_algos:
        div_str = f"{s['mean_diversity']:.3f}" if not np.isnan(s["mean_diversity"]) else "n/a"
        lines.append(f"{algo} & {s['mean_l2']:.6f} & {s['std_l2']:.6f} & {div_str} \\\\")
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")
    lines.append("")
    lines.append(r"\subsection{Convergence and Diversity Trajectories}")
    lines.append(
        r"Figure~\ref{fig:convergence} shows mean per-generation validation error, and "
        r"Figure~\ref{fig:diversity} shows the corresponding measured diversity trajectory "
        "for the same runs."
    )
    lines.append(r"\begin{figure}[h]")
    lines.append(r"\centering")
    lines.append(r"\includegraphics[width=\linewidth]{" + figures_rel + r"/convergence_comparison.png}")
    lines.append(r"\caption{Mean validation relative $L_2$ error per generation/iteration, by benchmark.}")
    lines.append(r"\label{fig:convergence}")
    lines.append(r"\end{figure}")
    lines.append(r"\begin{figure}[h]")
    lines.append(r"\centering")
    lines.append(r"\includegraphics[width=\linewidth]{" + figures_rel + r"/diversity_exploration_trajectories.png}")
    lines.append(r"\caption{Measured population/swarm/archive diversity per generation/iteration, by benchmark.}")
    lines.append(r"\label{fig:diversity}")
    lines.append(r"\end{figure}")
    lines.append("")
    lines.append(r"\section{Threats to Validity}")
    lines.append(
        f"(1) {sig_note} (2) Training runs on Colab CPU/GPU with a fixed step budget; "
        "results may shift under a larger step budget. (3) Only four PDE benchmarks with "
        "verified real trainers are included."
    )
    lines.append("")
    lines.append(r"\section{Reproducibility}")
    lines.append(
        r"All code is available at \texttt{https://github.com/Rahuldrabit/OptimizationOverviewPINN}. "
        "Every number in this paper is generated directly from the results JSON produced "
        "by this notebook."
    )
    lines.append("")
    lines.append(r"\end{document}")
    return "\n".join(lines)


results_file = os.path.join(config.output_dir, "hpo_comparison_results.json")
with open(results_file, "r") as f:
    results_json = _json.load(f)

stats = compute_algo_stats(results_json, algorithms)

paper_dir = os.path.join(config.output_dir, "paper")
os.makedirs(paper_dir, exist_ok=True)
paper_file = os.path.join(paper_dir, "nsys2026_paper.tex")

with open(paper_file, "w", encoding="utf-8") as f:
    f.write(build_latex(results_json, stats))

print(f"[OK] ACM sigconf paper generated: {paper_file}")
print("Compile it on Overleaf or a local TeX Live/MiKTeX install to get the PDF.")


## Step 8: Package and download results

In [ ]:
import shutil
from google.colab import files

zip_base = config.output_dir
shutil.make_archive(zip_base, "zip", ".", config.output_dir)
zip_path = f"{zip_base}.zip"

print(f"[+] Packaged: {zip_path}")
print("    Contents: hpo_comparison_results.json, MANUSCRIPT_REPORT.md, plots/, "
      "paper/nsys2026_paper.tex, runs/ (individual job results)")

files.download(zip_path)


---
## Next steps

1. **Compile the paper**: upload `paper/nsys2026_paper.tex` (from the downloaded zip) to
   [Overleaf](https://www.overleaf.com), or compile locally with `pdflatex` if you have
   a TeX Live / MiKTeX install. The ACM `acmart` class is available on Overleaf by default.
2. **Check every number** in the paper against `hpo_comparison_results.json` before
   submitting - the generator only writes numbers it computed from that file, but you
   should still sanity-check them yourself.
3. **Consider more seeds** if you want a real statistical significance claim (this
   notebook defaults to 2, which is descriptive-only; 10+ is a common bar for a
   paired significance test).
